# Three checks on B and N (inference only, no training): matching rule, test-vs-OOF gap, one fold vs the ensemble

**No training. About 20-30 minutes on one T4 GPU.**

1. **Matching rule.** The official-style F1 of the thesis matches predictions to lesions greedily by Dice. Here the same predictions are scored with two other rules: greedy by IoU with IoU >= 0.1 required, and the optimal one-to-one assignment (Hungarian) that maximises the total Dice. If the N - B difference keeps its size and sign, the thesis result does not depend on the matching rule I chose.
2. **Test-vs-OOF gap.** The test score of both models is above their OOF score. The scores are split by the number of lesions per image (1, 2, 3+) and by the total lesion area (three groups), and the OOF score is re-weighted to the test mix, to see how much of the gap is only the different mix of images.
3. **One fold model vs the ensemble.** Each OOF image is predicted by one fold model, each test image by the mean of five. Here every single fold model is scored on the test set at the same frozen post-processing, next to the 5-model mean.

**Inputs:** (1) ARCADE dataset (nirmalgaud), (2) the output of the first VG-DeepLab notebook (`dl3_r101_f0..4.pt`, `vg_dl3_f0..4.pt`). GPU = T4. In the CONFIG cell set `SMOKE_TEST = False`.

**Output:** folder `truef1_outputs` + `inference_checks_3a.zip`. Send the zip back.

In [1]:
# Do NOT let pip touch torch: --no-deps keeps the preinstalled torch exactly as it is.
!pip install -q --no-deps segmentation-models-pytorch
!pip install -q ultralytics
print("Installs complete.")

Installs complete.


In [2]:
import os, sys, re, json, math, time, shutil, zipfile, itertools, warnings, subprocess, traceback
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
from torch.amp import autocast
for _try in range(8):          # smp needs a few small packages; install any that are missing WITHOUT touching torch
    try:
        import segmentation_models_pytorch as smp; break
    except ModuleNotFoundError as e:
        print("missing package:", e.name, "-> installing (no deps)")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", e.name.split(".")[0].replace("_", "-")])
warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================
SMOKE_TEST = False         # True = quick plumbing check on 40 val + 40 test images (numbers are meaningless). False = the real run.
IMG_SIZE, STENOSIS_CATEGORY_ID = 512, 26
N_BOOT, N_BOOT_MAP = 10000, 200          # bootstrap resamples (means / mAP)
HEADLINE = "balanced-DEPLOY"
YOLO_CONF, YOLO_MAX_DET = 0.001, 20      # same as the YOLOv8s-seg notebook
CAND_THR, CAND_MIN_AREA, CAND_MAX = 0.30, 100, 8      # candidate list used for mAP of the semantic models (same as the earlier notebooks)
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu"); USE_CUDA = DEVICE.type == "cuda"
ON_KAGGLE = Path("/kaggle/working").exists()
BASE = Path("/kaggle/working") if ON_KAGGLE else (Path("/content") if Path("/content").exists() else Path.cwd())
OUT_DIR = BASE / "truef1_outputs"; OUT_DIR.mkdir(parents=True, exist_ok=True)
SEARCH_ROOTS = [Path(p) for p in os.environ.get("UNI_WEIGHT_ROOTS", "/kaggle/input:/content/weights:/content/drive/MyDrive/arcade_stenosis_runs").split(":")]
N_BOOT_MAP_BIG = 10000                   # mAP resamples for the Bonferroni intervals (2000 is faster)
ALPHA_BONF = 0.05 / 15                   # 15 pairs of six models
if SMOKE_TEST: N_BOOT, N_BOOT_MAP, N_BOOT_MAP_BIG = 500, 20, 50
ERRORS = []
print("Torch", torch.__version__, "| smp", smp.__version__, "|", torch.cuda.get_device_name(0) if USE_CUDA else "NO GPU (enable a GPU in the notebook settings!)", "| SMOKE_TEST =", SMOKE_TEST)

# frozen operating points of the two 5-fold ensembles (chosen on out-of-fold predictions in the first VG-DeepLab run: frozen_choices_v3.json)
FROZEN_OPS = {
    "B": {"balanced-DEPLOY": dict(thr=0.40, min_area=150, tau=0.5, rel=0.75, max_k=99, top1=False),
          "maxF1-ARCADE":    dict(thr=0.40, min_area=331, tau=0.6, rel=0.75, max_k=3,  top1=True)},
    "N": {"balanced-DEPLOY": dict(thr=0.35, min_area=150, tau=0.2, rel=0.75, max_k=3,  top1=False),
          "maxF1-ARCADE":    dict(thr=0.45, min_area=150, tau=0.7, rel=0.0,  max_k=3,  top1=True)},
}

Torch 2.11.0+cu130 | smp 0.6.0 | Tesla T4 | SMOKE_TEST = False


In [3]:
# ============================================================
# ARCADE data + instance / metric code (identical logic to the VG-DeepLab and YOLOv8s-seg notebooks)
# ============================================================
def find_arcade_root():
    for root in [Path(p) for p in os.environ.get("UNI_DATA_ROOTS", "/kaggle/input").split(":")] + [Path.home() / ".cache" / "kagglehub"]:
        if root.exists():
            c = [p for p in root.rglob("train.json") if "stenosis" in str(p).lower()]
            if c: return c[0].parent.parent.parent
    return None

DATASET_ROOT = find_arcade_root()
if DATASET_ROOT is None:
    raise RuntimeError("ARCADE dataset not found. In Kaggle: Add Input -> search 'arcade-dataset' (nirmalgaud) -> Add, then Save Version -> Save & Run All.")
for s in ["train", "val", "test"]:
    assert (DATASET_ROOT / s / "annotations" / f"{s}.json").exists() and (DATASET_ROOT / s / "images").exists()
print("ARCADE root:", DATASET_ROOT)

class Inst:
    """One instance mask, cropped to its bounding box."""
    __slots__ = ("y0", "x0", "m", "area", "score", "extra")
    def __init__(self, y0, x0, m, score=1.0, extra=None):
        self.y0, self.x0, self.m = int(y0), int(x0), m
        self.area = int(m.sum()); self.score, self.extra = float(score), extra
    @property
    def y1(self): return self.y0 + self.m.shape[0]
    @property
    def x1(self): return self.x0 + self.m.shape[1]

def inst_from_full(full_bool, score=1.0):
    ys, xs = np.nonzero(full_bool)
    if len(ys) == 0: return None
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    return Inst(y0, x0, full_bool[y0:y1, x0:x1].copy(), score)

def inst_inter(a, b):
    y0, y1 = max(a.y0, b.y0), min(a.y1, b.y1); x0, x1 = max(a.x0, b.x0), min(a.x1, b.x1)
    if y0 >= y1 or x0 >= x1: return 0
    return int(np.count_nonzero(a.m[y0 - a.y0:y1 - a.y0, x0 - a.x0:x1 - a.x0] & b.m[y0 - b.y0:y1 - b.y0, x0 - b.x0:x1 - b.x0]))

def inst_dice(a, b):
    s = a.area + b.area
    return 0.0 if s == 0 else 2.0 * inst_inter(a, b) / s

def inst_iou(a, b):
    i = inst_inter(a, b); u = a.area + b.area - i
    return 0.0 if u == 0 else i / u

def box_iou(a, b):
    ih, iw = min(a.y1, b.y1) - max(a.y0, b.y0), min(a.x1, b.x1) - max(a.x0, b.x0)
    if ih <= 0 or iw <= 0: return 0.0
    i = ih * iw; u = (a.y1 - a.y0) * (a.x1 - a.x0) + (b.y1 - b.y0) * (b.x1 - b.x0) - i
    return i / u

def paint(insts, shape):
    canvas = np.zeros(shape, bool)
    for i in insts: canvas[i.y0:i.y1, i.x0:i.x1] |= i.m
    return canvas

def greedy_match(preds, gts, fn):
    pairs = []
    for i, p in enumerate(preds):
        for j, g in enumerate(gts):
            s = fn(p, g)
            if s > 0: pairs.append((s, i, j))
    pairs.sort(key=lambda t: -t[0])
    up, ug, out = set(range(len(preds))), set(range(len(gts))), []
    for s, i, j in pairs:
        if i in up and j in ug: out.append((i, j, s)); up.discard(i); ug.discard(j)
    return out, len(up), len(ug)

def inst_greedy_match(preds, gts, criterion="dice"):
    return greedy_match(preds, gts, inst_dice if criterion == "dice" else inst_iou)

def p1_image(preds, gts):
    """Official-style ARCADE F1 for one image: matched pairs score their Dice, every unmatched prediction or lesion scores 0."""
    if not preds and not gts: return 1.0
    pairs, n_up, n_ug = inst_greedy_match(preds, gts, "dice")
    return float(np.mean([s for _, _, s in pairs] + [0.0] * (n_up + n_ug)))

def full_metrics(preds_by_img, items):
    p1, dimg, iimg = [], [], []
    TP = FP = FN = pTP = pFP = pFN = 0
    for k, it in enumerate(items):
        preds, gts = preds_by_img[k], it["gts"]
        p1.append(p1_image(preds, gts))
        pairs, n_up, n_ug = inst_greedy_match(preds, gts, "iou")
        tp = sum(1 for _, _, s in pairs if s >= 0.5)
        TP += tp; FP += len(preds) - tp; FN += len(gts) - tp
        pr, gt = paint(preds, it["shape"]), paint(gts, it["shape"])
        a, b, c = np.count_nonzero(pr & gt), np.count_nonzero(pr & ~gt), np.count_nonzero(~pr & gt)
        pTP += a; pFP += b; pFN += c
        dimg.append(1.0 if a + b + c == 0 else 2 * a / (2 * a + b + c))
        iimg.append(1.0 if a + b + c == 0 else a / (a + b + c))
    P, R = TP / max(TP + FP, 1e-9), TP / max(TP + FN, 1e-9)
    return {"P1": float(np.mean(p1)), "F1": 2 * P * R / max(P + R, 1e-9), "Precision": P, "Recall": R, "TP": TP, "FP": FP, "FN": FN,
            "Dice": 2 * pTP / max(2 * pTP + pFP + pFN, 1e-9), "IoU": pTP / max(pTP + pFP + pFN, 1e-9),
            "Dice_img": float(np.mean(dimg)), "Dice_img_std": float(np.std(dimg)), "IoU_img": float(np.mean(iimg)),
            "pred_per_img": (TP + FP) / len(items), "per_image_P1": np.array(p1), "per_image_Dice": np.array(dimg)}

_g = [Inst(0, 0, np.ones((5, 5), bool))]
assert p1_image(_g, _g) == 1.0 and abs(p1_image(_g + [Inst(9, 9, np.ones((3, 3), bool))], _g) - 0.5) < 1e-9
print("self-check official-style F1: perfect = 1.0 | one extra false positive = 0.5  -> OK")

def polys_to_mask(segs, h, w):
    full = np.zeros((h, w), np.uint8)
    for poly in segs or []:
        if isinstance(poly, list) and len(poly) >= 6:
            cv2.fillPoly(full, [np.round(np.array(poly, np.float32).reshape(-1, 2)).astype(np.int32)], 1)
    return full

def load_split(split, with_images=True):
    coco = json.load(open(DATASET_ROOT / split / "annotations" / f"{split}.json"))
    anns = {}
    for a in coco["annotations"]:
        if a["category_id"] == STENOSIS_CATEGORY_ID: anns.setdefault(a["image_id"], []).append(a)
    imgs, masks, items = [], [], []
    for im in tqdm(sorted(coco["images"], key=lambda d: d["id"]), desc=f"ARCADE {split}", mininterval=5):
        h, w = im["height"], im["width"]
        if with_images:
            g = cv2.imread(str(DATASET_ROOT / split / "images" / im["file_name"]), cv2.IMREAD_GRAYSCALE)
            assert g.shape == (IMG_SIZE, IMG_SIZE), f"{im['file_name']} is {g.shape}, expected 512x512 (ARCADE native)"
            imgs.append(g)
        gts, union = [], np.zeros((h, w), np.uint8)
        for a in anns.get(im["id"], []):
            m = polys_to_mask(a.get("segmentation"), h, w); union |= m
            gi = inst_from_full(m.astype(bool))
            if gi is not None: gts.append(gi)
        masks.append(union); items.append({"file": im["file_name"], "shape": (h, w), "gts": gts})
    return (np.stack(imgs) if with_images else None), np.stack(masks), items

TRAIN_IMGS, TRAIN_MASKS, TRAIN_ITEMS = load_split("train")
VAL_IMGS, VAL_MASKS, VAL_ITEMS = load_split("val")
TEST_IMGS, TEST_MASKS, TEST_ITEMS = load_split("test")
MIN_AREA_REF = int(np.percentile([g.area for it in TRAIN_ITEMS + VAL_ITEMS for g in it["gts"]], 5))   # same definition as the VG-DeepLab notebook (train+val)
# the colour read used by the original U-Net / DeepLabV3+ / YOLO notebooks must equal 3 x the grey read used here
_f = DATASET_ROOT / "test" / "images" / TEST_ITEMS[0]["file"]; _c = cv2.imread(str(_f), cv2.IMREAD_COLOR)
GRAY_OK = bool((_c[..., 0] == _c[..., 1]).all() and (_c[..., 1] == _c[..., 2]).all() and (_c[..., 0] == TEST_IMGS[0]).all())
print("images are pure grey (colour read == grey read):", GRAY_OK)
if SMOKE_TEST:
    rs = np.random.default_rng(0)
    vi = np.sort(rs.choice(len(VAL_IMGS), min(40, len(VAL_IMGS)), replace=False)); ti = np.sort(rs.choice(len(TEST_IMGS), min(40, len(TEST_IMGS)), replace=False))
    VAL_IMGS, VAL_MASKS, VAL_ITEMS = VAL_IMGS[vi], VAL_MASKS[vi], [VAL_ITEMS[i] for i in vi]
    TEST_IMGS, TEST_MASKS, TEST_ITEMS = TEST_IMGS[ti], TEST_MASKS[ti], [TEST_ITEMS[i] for i in ti]
GTU = {"val": VAL_MASKS > 0, "test": TEST_MASKS > 0}; IMGS = {"val": VAL_IMGS, "test": TEST_IMGS}; ITEMS = {"val": VAL_ITEMS, "test": TEST_ITEMS}
N_TEST = len(TEST_ITEMS)
for s in ["val", "test"]:
    n = sum(len(i["gts"]) for i in ITEMS[s]); print(f"{s:5s}: {len(ITEMS[s])} images | {n} lesions | {n / len(ITEMS[s]):.2f} per image | images without a lesion: {sum(1 for i in ITEMS[s] if not i['gts'])}")
print("5th-percentile lesion area (train+val) =", MIN_AREA_REF, "px")

ARCADE root: /kaggle/input/datasets/nirmalgaud/arcade-dataset/arcade/stenosis
self-check official-style F1: perfect = 1.0 | one extra false positive = 0.5  -> OK


ARCADE train:   0%|          | 0/1000 [00:00<?, ?it/s]

ARCADE val:   0%|          | 0/200 [00:00<?, ?it/s]

ARCADE test:   0%|          | 0/300 [00:00<?, ?it/s]

images are pure grey (colour read == grey read): True
val  : 200 images | 406 lesions | 2.03 per image | images without a lesion: 0
test : 300 images | 386 lesions | 1.29 per image | images without a lesion: 0
5th-percentile lesion area (train+val) = 331 px


In [4]:
# ============================================================
# Candidates, post-processing search, operating points (same code as the VG-DeepLab notebook)
# ============================================================
def cand_dict(preds, gts, gtu_k):
    preds = sorted(preds, key=lambda t: -t.score)
    D = np.zeros((len(preds), len(gts)), np.float32)
    for i, pr in enumerate(preds):
        for j, g in enumerate(gts): D[i, j] = inst_dice(pr, g)
    inter = np.array([np.count_nonzero(pr.m & gtu_k[pr.y0:pr.y1, pr.x0:pr.x1]) for pr in preds], np.int64)
    return dict(preds=preds, scores=np.array([pr.score for pr in preds]), area=np.array([pr.area for pr in preds], np.int64),
                inter=inter, D=D, gt_area=int(gtu_k.sum()), ngt=len(gts))

def build_cands(probs, gtu, items, thr, min_area):
    """probs: (N,H,W) float; one dict per image with connected components (score = mean probability) sorted by score."""
    out = []
    for k in range(len(probs)):
        p = probs[k].astype(np.float32)
        n, lab, stats, _ = cv2.connectedComponentsWithStats((p > thr).astype(np.uint8), connectivity=8)
        preds = []
        for i in range(1, n):
            x, y, w, h, a = stats[i]
            if a < min_area: continue
            m = lab[y:y + h, x:x + w] == i
            preds.append(Inst(y, x, m, score=float(p[y:y + h, x:x + w][m].mean())))
        out.append(cand_dict(preds, items[k]["gts"], gtu[k]))
    return out

def keep_idx(sc, tau, rel, max_k, top1):
    n = len(sc)
    keep = [i for i in range(n) if sc[i] >= tau and sc[i] >= rel * sc[0]][:max_k]
    if not keep and top1 and n: keep = [0]
    return keep

def p1_from_D(Dsub, n_gt):
    n_pred = Dsub.shape[0]
    if n_pred == 0 and n_gt == 0: return 1.0
    scores, matched_p, matched_g = [], set(), set()
    if n_pred and n_gt:
        ii, jj = np.nonzero(Dsub > 0)
        for o in np.argsort(-Dsub[ii, jj], kind="stable"):
            i, j = int(ii[o]), int(jj[o])
            if i not in matched_p and j not in matched_g:
                scores.append(float(Dsub[i, j])); matched_p.add(i); matched_g.add(j)
    scores += [0.0] * ((n_pred - len(matched_p)) + (n_gt - len(matched_g)))
    return float(np.mean(scores))

def eval_cfg(C, tau, rel, max_k, top1):
    d_sum = p_sum = 0.0
    for c in C:
        keep = keep_idx(c["scores"], tau, rel, max_k, top1)
        P = int(c["area"][keep].sum()) if keep else 0; I = int(c["inter"][keep].sum()) if keep else 0
        d_sum += 1.0 if P + c["gt_area"] == 0 else 2.0 * I / (P + c["gt_area"])
        p_sum += p1_from_D(c["D"][keep] if keep else np.zeros((0, c["ngt"]), np.float32), c["ngt"])
    return d_sum / len(C), p_sum / len(C)

def select_insts(c, cfg):
    return [c["preds"][i] for i in keep_idx(c["scores"], cfg["tau"], cfg["rel"], cfg["max_k"], cfg["top1"])]

GRID_SEM = dict(thr=[0.30, 0.35, 0.40, 0.45, 0.50, 0.55], min_area=sorted({150, MIN_AREA_REF}),
                tau=[0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8], rel=[0.0, 0.5, 0.75], max_k=[1, 2, 3, 99], top1=[True, False])
GRID_YOLO = dict(thr=[0.0], min_area=[0], tau=[0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8], rel=[0.0, 0.5, 0.75], max_k=[1, 2, 3, 99], top1=[True, False])

def search_postproc(cands_fn, grid):
    """cands_fn(thr, min_area) -> candidate dicts of the VALIDATION images."""
    rows = []
    for thr, ma in itertools.product(grid["thr"], grid["min_area"]):
        C = cands_fn(thr, ma)
        for tau, rel, mk, t1 in itertools.product(grid["tau"], grid["rel"], grid["max_k"], grid["top1"]):
            d, p = eval_cfg(C, tau, rel, mk, t1)
            rows.append(dict(thr=thr, min_area=ma, tau=tau, rel=rel, max_k=mk, top1=t1, dice_img=d, p1=p))
    return pd.DataFrame(rows)

def pick_ops(df):
    ops = {}
    for tag, sub in [("ARCADE", df), ("DEPLOY", df[~df.top1])]:
        for obj, key in [("maxDice", df.dice_img), ("maxF1", df.p1), ("balanced", (df.dice_img + df.p1) / 2)]:
            r = df.loc[key.loc[sub.index].idxmax()]
            ops[f"{obj}-{tag}"] = dict(thr=float(r.thr), min_area=int(r.min_area), tau=float(r.tau), rel=float(r.rel),
                                       max_k=int(r.max_k), top1=bool(r.top1), val_dice=float(r.dice_img), val_p1=float(r.p1))
    return ops

def pixel_confusion(preds_by_img, items):
    TP = FP = FN = total = 0
    for preds, it in zip(preds_by_img, items):
        h, w = it["shape"]; total += h * w
        pr, gt = paint(preds, it["shape"]), paint(it["gts"], it["shape"])
        TP += int(np.count_nonzero(pr & gt)); FP += int(np.count_nonzero(pr & ~gt)); FN += int(np.count_nonzero(~pr & gt))
    TN = total - TP - FP - FN
    return dict(TP=TP, FP=FP, FN=FN, TN=TN, precision=TP / max(TP + FP, 1), recall=TP / max(TP + FN, 1), specificity=TN / max(TN + FP, 1),
                accuracy=(TP + TN) / max(total, 1), dice=2 * TP / max(2 * TP + FP + FN, 1), iou=TP / max(TP + FP + FN, 1))

def match_counts(preds_by_img, items, thr, fn=inst_iou):
    """Per image: (true positives at overlap >= thr, number of predictions, number of GT lesions)."""
    tp, npred, ngt = [], [], []
    for preds, it in zip(preds_by_img, items):
        pairs, _, _ = greedy_match(preds, it["gts"], fn)
        tp.append(sum(1 for _, _, s in pairs if s >= thr)); npred.append(len(preds)); ngt.append(len(it["gts"]))
    return np.array(tp), np.array(npred), np.array(ngt)

def f1_from_counts(cnt):
    tp, npred, ngt = cnt
    return 2 * tp.sum() / max(npred.sum() + ngt.sum(), 1)

def cap_cands(C, max_per_img):
    out = []
    for c in C:
        n = min(len(c["preds"]), max_per_img)
        out.append(dict(preds=c["preds"][:n], scores=c["scores"][:n], area=c["area"][:n], inter=c["inter"][:n], D=c["D"][:n], gt_area=c["gt_area"], ngt=c["ngt"]))
    return out

def map_prep(C, items):
    ious = []
    for c, it in zip(C, items):
        m = np.zeros((len(c["preds"]), len(it["gts"])), np.float32)
        for i, p in enumerate(c["preds"]):
            for j, g in enumerate(it["gts"]): m[i, j] = inst_iou(p, g)
        ious.append(m)
    return ious, np.array([len(it["gts"]) for it in items]), [np.asarray(c["scores"], np.float64) for c in C]

def map_from_prep(prep, idx=None, thrs=np.arange(0.50, 0.96, 0.05)):
    """COCO-style mask AP (101-point). idx = optional bootstrap resample of images. Returns (AP50, AP50-95)."""
    ious, ngt, scores = prep
    if idx is None: idx = np.arange(len(ngt))
    n_gt = int(ngt[idx].sum())
    order = sorted(((float(scores[k][i]), j, i) for j, k in enumerate(idx) for i in range(len(scores[k]))), key=lambda d: -d[0])
    aps = []
    for thr in thrs:
        used = [np.zeros(ngt[k], bool) for k in idx]; tp = np.zeros(len(order))
        for n, (_, j, i) in enumerate(order):
            k = idx[j]
            if ngt[k] == 0: continue
            cand = np.where(~used[j], ious[k][i], -1.0); g = int(cand.argmax())
            if cand[g] >= thr: used[j][g] = True; tp[n] = 1
        ctp, cfp = np.cumsum(tp), np.cumsum(1 - tp); rec = ctp / max(n_gt, 1)
        prec = np.maximum.accumulate((ctp / np.maximum(ctp + cfp, 1e-9))[::-1])[::-1] if len(order) else np.zeros(0)
        aps.append(float(np.mean([prec[rec >= r].max() if np.any(rec >= r) else 0.0 for r in np.linspace(0, 1, 101)])))
    return aps[0], float(np.mean(aps))

# ---- bootstrap over test images (the same resamples for every model -> paired comparisons) ----
BI = np.random.default_rng(0).integers(0, N_TEST, (N_BOOT, N_TEST))
BI_MAP = BI[:N_BOOT_MAP]
def boot_mean(v): return v[BI].mean(1)
def boot_f1(cnt):
    tp, npred, ngt = cnt
    return 2 * tp[BI].sum(1) / np.maximum(npred[BI].sum(1) + ngt[BI].sum(1), 1)
def ci95(b): return np.percentile(b, [2.5, 97.5])
def fmt_ci(v, b): lo, hi = ci95(b); return f"{v:.4f} [{lo:.3f},{hi:.3f}]"
print("metric code ready | bootstrap resamples:", N_BOOT, "(means, F1) /", N_BOOT_MAP, "(mAP)")

metric code ready | bootstrap resamples: 10000 (means, F1) / 200 (mAP)


In [5]:
class VGNet(nn.Module):
    """5-fold models (VG-DeepLab notebook). aux: vessel channel. refine: vessel-guided refinement head. forward -> (stenosis logit, coarse logit, vessel logit or None)."""
    def __init__(self, aux=True, refine=True):
        super().__init__()
        self.aux, self.refine = aux, refine
        self.net = smp.DeepLabV3Plus(encoder_name="resnet101", encoder_weights=None, in_channels=3, classes=2 if aux else 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        if refine:
            cin = 3 if aux else 2
            self.ref = nn.Sequential(nn.Conv2d(cin, 24, 3, padding=1), nn.ReLU(inplace=True),
                                     nn.Conv2d(24, 24, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
                                     nn.Conv2d(24, 24, 3, padding=4, dilation=4), nn.ReLU(inplace=True),
                                     nn.Conv2d(24, 24, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(24, 1, 1))
    def forward(self, x):
        out = self.net((x - self.mean) / self.std)
        s = out[:, :1]; v = out[:, 1:2] if self.aux else None
        if not self.refine: return s, s, v
        f = [x[:, :1] * 2 - 1, torch.sigmoid(s.detach()) * 2 - 1]
        if self.aux: f.append(torch.sigmoid(v.detach()) * 2 - 1)
        return s + self.ref(torch.cat(f, 1)), s, v

def n_params(m): return sum(p.numel() for p in m.parameters()) / 1e6

def to_x(imgs):
    return torch.from_numpy(imgs).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1).to(DEVICE)      # grey image in 3 identical channels, [0,1]


@torch.no_grad()
def predict_ms(model, imgs, bs=8):
    """5-fold models: test-time augmentation 'ms' (2 scales 1.0 / 1.25 x horizontal flip), exactly as in the VG-DeepLab notebook."""
    model.eval(); S = []
    for st in range(0, len(imgs), bs):
        x = to_x(imgs[st:st + bs]); acc = 0.0
        for sc in (1.0, 1.25):
            xs = x if sc == 1.0 else F.interpolate(x, scale_factor=sc, mode="bilinear", align_corners=False)
            for flip in (False, True):
                xi = torch.flip(xs, dims=[3]) if flip else xs
                with autocast("cuda", enabled=USE_CUDA): s2, _, _ = model(xi)
                o = torch.sigmoid(s2.float())
                if flip: o = torch.flip(o, dims=[3])
                if sc != 1.0: o = F.interpolate(o, size=x.shape[-2:], mode="bilinear", align_corners=False)
                acc = acc + o
        S.append((acc / 4)[:, 0].cpu().numpy().astype(np.float16))
    return np.concatenate(S)


In [6]:
# ============================================================
# Pool (train + val = 1200 images), the 5 folds, and the fold weights
# ============================================================
from sklearn.model_selection import StratifiedKFold
POOL_IMGS = np.concatenate([TRAIN_IMGS, VAL_IMGS]); POOL_MASKS = np.concatenate([TRAIN_MASKS, VAL_MASKS]); POOL_ITEMS = TRAIN_ITEMS + VAL_ITEMS
GTU_POOL = POOL_MASKS > 0; N_POOL = len(POOL_ITEMS)
assert N_POOL == 1200, f"expected 1200 pooled images, got {N_POOL}"
y_strat = np.array([min(len(it["gts"]), 3) for it in POOL_ITEMS])
FOLDS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(np.zeros(N_POOL), y_strat))
print("fold sizes (held-out images):", [len(va) for _, va in FOLDS], "| lesions in pool:", sum(len(it["gts"]) for it in POOL_ITEMS))

def find_fold_ckpts(prefix):
    found = {}
    for root in [Path("/kaggle/input"), Path("/content"), BASE]:
        if root.exists():
            for p in root.rglob(f"{prefix}_f[0-4].pt"):
                f = int(p.stem.split("_f")[-1]); found.setdefault(f, p)
    assert sorted(found) == [0, 1, 2, 3, 4], f"{prefix}_f0..4.pt not all found (found folds: {sorted(found)}). Add the output of the first VG-DeepLab notebook as an input."
    return found
CK = {"B": find_fold_ckpts("dl3_r101"), "N": find_fold_ckpts("vg_dl3")}
for k, v in CK.items(): print(k, {f: str(p) for f, p in v.items()})


fold sizes (held-out images): [240, 240, 240, 240, 240] | lesions in pool: 2031
B {0: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/dl3_r101_f0.pt', 1: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/dl3_r101_f1.pt', 2: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/dl3_r101_f2.pt', 3: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/dl3_r101_f3.pt', 4: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/dl3_r101_f4.pt'}
N {0: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/vg_dl3_f0.pt', 1: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/vg_dl3_f1.pt', 2: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/vg_dl3_f2.pt', 3: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/vg_dl3_f3.pt', 4: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run

In [7]:
# ============================================================
# Probability maps: OOF (each pooled image by the fold model that did not train on it) and test (mean of the 5 fold models)
# ============================================================
def oof_and_test(key, aux, refine):
    oof = np.zeros((N_POOL, IMG_SIZE, IMG_SIZE), np.float16); acc = np.zeros(TEST_IMGS.shape, np.float32); t0 = time.time()
    for f, (tr, va) in enumerate(FOLDS):
        net = VGNet(aux, refine); net.load_state_dict(torch.load(CK[key][f], map_location="cpu", weights_only=True), strict=True); net.to(DEVICE).eval()
        oof[va] = predict_ms(net, POOL_IMGS[va]); acc += predict_ms(net, TEST_IMGS).astype(np.float32); del net
        if USE_CUDA: torch.cuda.empty_cache()
        print(f"{key}: fold {f + 1}/5 done | {time.time() - t0:.0f}s")
    return oof, (acc / 5).astype(np.float16)
OOF, TEST = {}, {}
OOF["B"], TEST["B"] = oof_and_test("B", False, False)
OOF["N"], TEST["N"] = oof_and_test("N", True, True)
GTU_TEST = TEST_MASKS > 0
print("maps ready")

B: fold 1/5 done | 55s
B: fold 2/5 done | 107s
B: fold 3/5 done | 163s
B: fold 4/5 done | 218s
B: fold 5/5 done | 274s
N: fold 1/5 done | 69s
N: fold 2/5 done | 137s
N: fold 3/5 done | 206s
N: fold 4/5 done | 274s
N: fold 5/5 done | 343s
maps ready


In [8]:
# ============================================================
# True F1 per image, scoring helper, and a reproduction check against the thesis
# ============================================================
def truef1_image(preds, gts):
    """Usual F1 form with Dice credit: 2 * sum(matched Dice) / (2M + unmatched predictions + unmatched lesions)."""
    if not preds and not gts: return 1.0
    pairs, n_up, n_ug = inst_greedy_match(preds, gts, "dice")
    return 2.0 * sum(s for _, _, s in pairs) / (2 * len(pairs) + n_up + n_ug)
_g = [Inst(0, 0, np.ones((5, 5), bool))]
assert truef1_image(_g, _g) == 1.0 and abs(truef1_image(_g + [Inst(9, 9, np.ones((3, 3), bool))], _g) - 2 / 3) < 1e-9
print("self-check true F1: perfect = 1.0 | one extra false positive = 0.667 (official-style gives 0.5) -> OK")

def score(probs, gtu, items, op):
    C = build_cands(probs, gtu, items, op["thr"], op["min_area"]); preds = [select_insts(c, op) for c in C]
    fm = full_metrics(preds, items)
    return dict(preds=preds, dice=fm["per_image_Dice"], p1=fm["per_image_P1"], tf1=np.array([truef1_image(p, it["gts"]) for p, it in zip(preds, items)]),
                cnt=match_counts(preds, items, 0.5), npred=np.array([len(p) for p in preds]))

DATA = {"test": (TEST, GTU_TEST, TEST_ITEMS), "oof": (OOF, GTU_POOL, POOL_ITEMS)}
OWN = {}
for split, (maps, gtu, items) in DATA.items():
    for key in ["B", "N"]:
        for opn in ["balanced-DEPLOY", "maxF1-ARCADE"]:
            OWN[(split, key, opn)] = score(maps[key], gtu, items, FROZEN_OPS[key][opn])
EXPECTED = {("test", "B", "balanced-DEPLOY"): (0.6197, 0.5165), ("test", "N", "balanced-DEPLOY"): (0.6358, 0.5436),
            ("test", "B", "maxF1-ARCADE"): (None, 0.5230), ("test", "N", "maxF1-ARCADE"): (None, 0.5579),
            ("oof", "B", "balanced-DEPLOY"): (0.5941, 0.4653), ("oof", "N", "balanced-DEPLOY"): (0.6028, 0.4631)}
rows = []
for k, (ed, ep) in EXPECTED.items():
    s = OWN[k]
    for m, e, got in [("Dice/img", ed, s["dice"].mean()), ("official-style F1", ep, s["p1"].mean())]:
        if e is None: continue
        rows.append(dict(split=k[0], model=k[1], op=k[2], metric=m, thesis=e, here=round(float(got), 4), diff=round(float(got - e), 4), status="OK" if abs(got - e) <= 0.005 else "MISMATCH"))
REPRO = pd.DataFrame(rows); REPRO.to_csv(OUT_DIR / "reproduction_check.csv", index=False); print(REPRO.to_string(index=False))
if (REPRO.status == "MISMATCH").any(): print("\nWARNING: a number does not reproduce - send the output before using anything below.")

self-check true F1: perfect = 1.0 | one extra false positive = 0.667 (official-style gives 0.5) -> OK
split model              op            metric  thesis   here  diff status
 test     B balanced-DEPLOY          Dice/img  0.6197 0.6197   0.0     OK
 test     B balanced-DEPLOY official-style F1  0.5165 0.5165  -0.0     OK
 test     N balanced-DEPLOY          Dice/img  0.6358 0.6358  -0.0     OK
 test     N balanced-DEPLOY official-style F1  0.5436 0.5436  -0.0     OK
 test     B    maxF1-ARCADE official-style F1  0.5230 0.5230   0.0     OK
 test     N    maxF1-ARCADE official-style F1  0.5579 0.5579  -0.0     OK
  oof     B balanced-DEPLOY          Dice/img  0.5941 0.5941  -0.0     OK
  oof     B balanced-DEPLOY official-style F1  0.4653 0.4653   0.0     OK
  oof     N balanced-DEPLOY          Dice/img  0.6028 0.6028  -0.0     OK
  oof     N balanced-DEPLOY official-style F1  0.4631 0.4631  -0.0     OK


In [9]:
# ============================================================
# 1) True F1 next to the official-style F1 (own frozen operating points)
# ============================================================
BI_T = np.random.default_rng(0).integers(0, len(TEST_ITEMS), (N_BOOT, len(TEST_ITEMS)))
BI_O = np.random.default_rng(1).integers(0, N_POOL, (N_BOOT, N_POOL))
BIX = {"test": BI_T, "oof": BI_O}
def iv(b, a=0.05): return np.percentile(b, [100 * a / 2, 100 * (1 - a / 2)])
rows = []
for (split, key, opn), s in OWN.items():
    bi = BIX[split]
    for m in ["p1", "tf1"]:
        v = s[m]; lo, hi = iv(v[bi].mean(1))
        rows.append(dict(split=split, model=key, operating_point=opn, metric={"p1": "official-style F1", "tf1": "true F1 (Dice credit)"}[m],
                         value=round(float(v.mean()), 4), ci95=f"[{lo:.3f}, {hi:.3f}]"))
    rows.append(dict(split=split, model=key, operating_point=opn, metric="images with exactly 1 prediction and 1 lesion",
                     value=int(((s["npred"] == 1) & (np.array([len(it["gts"]) for it in DATA[split][2]]) == 1)).sum()), ci95=""))
T1 = pd.DataFrame(rows); T1.to_csv(OUT_DIR / "T1_true_f1.csv", index=False); print(T1.to_string(index=False))

split model operating_point                                        metric    value           ci95
 test     B balanced-DEPLOY                             official-style F1   0.5165 [0.489, 0.544]
 test     B balanced-DEPLOY                         true F1 (Dice credit)   0.5587 [0.535, 0.583]
 test     B balanced-DEPLOY images with exactly 1 prediction and 1 lesion 159.0000               
 test     B    maxF1-ARCADE                             official-style F1   0.5230 [0.496, 0.551]
 test     B    maxF1-ARCADE                         true F1 (Dice credit)   0.5621 [0.538, 0.586]
 test     B    maxF1-ARCADE images with exactly 1 prediction and 1 lesion 170.0000               
 test     N balanced-DEPLOY                             official-style F1   0.5436 [0.515, 0.572]
 test     N balanced-DEPLOY                         true F1 (Dice credit)   0.5798 [0.555, 0.605]
 test     N balanced-DEPLOY images with exactly 1 prediction and 1 lesion 167.0000               
 test     N    maxF1

In [10]:
# ============================================================
# CHECK 1) Matching rule: same predictions, three ways of pairing predictions with lesions
# ============================================================
from scipy.optimize import linear_sum_assignment
def p1_rule(preds, gts, rule):
    """Official-style F1 of one image = (sum of matched Dice) / (matched pairs + unmatched predictions + unmatched lesions)."""
    if not preds and not gts: return 1.0
    if not preds or not gts: return 0.0
    D = np.array([[inst_dice(p, g) for g in gts] for p in preds])
    if rule == "greedy Dice (thesis)":
        pairs, up, ug = inst_greedy_match(preds, gts, "dice"); cred = [s for _, _, s in pairs]; M = len(pairs)
    elif rule == "greedy IoU >= 0.1":
        pairs, up, ug = inst_greedy_match(preds, gts, "iou"); keep = [(i, j) for i, j, s in pairs if s >= 0.1]
        cred = [D[i, j] for i, j in keep]; M = len(keep)
    elif rule == "Hungarian (max total Dice)":
        r, c = linear_sum_assignment(-D); keep = [(i, j) for i, j in zip(r, c) if D[i, j] > 0]
        cred = [D[i, j] for i, j in keep]; M = len(keep)
    return float(sum(cred)) / (M + (len(preds) - M) + (len(gts) - M))
RULES = ["greedy Dice (thesis)", "greedy IoU >= 0.1", "Hungarian (max total Dice)"]
rows = []
for split, (maps, gtu, items) in DATA.items():
    bi = BIX[split]; per = {}
    for key in ["B", "N"]:
        pr = OWN[(split, key, "balanced-DEPLOY")]["preds"]
        for rule in RULES: per[(key, rule)] = np.array([p1_rule(p, it["gts"], rule) for p, it in zip(pr, items)])
    for rule in RULES:
        b = per[("N", rule)][bi].mean(1) - per[("B", rule)][bi].mean(1); lo, hi = iv(b, 0.05)
        rows.append(dict(split=split, rule=rule, B=round(float(per[("B", rule)].mean()), 4), N=round(float(per[("N", rule)].mean()), 4),
                         N_minus_B=round(float(per[("N", rule)].mean() - per[("B", rule)].mean()), 4), ci95=f"[{lo:+.4f}, {hi:+.4f}]", excl0=bool(lo > 0 or hi < 0)))
T3 = pd.DataFrame(rows); T3.to_csv(OUT_DIR / "C1_matching_rule.csv", index=False); print(T3.to_string(index=False))
chk = T3[(T3.rule == RULES[0]) & (T3.split == "test")].iloc[0]
print(f"\nself-check: greedy-Dice rule on test gives B = {chk.B}, N = {chk.N} (thesis: 0.5165 and 0.5436) ->", "OK" if abs(chk.B - 0.5165) < 0.003 and abs(chk.N - 0.5436) < 0.003 or SMOKE_TEST else "MISMATCH")

split                       rule      B      N  N_minus_B               ci95  excl0
 test       greedy Dice (thesis) 0.5165 0.5436     0.0271 [+0.0128, +0.0421]   True
 test          greedy IoU >= 0.1 0.5159 0.5430     0.0271 [+0.0128, +0.0421]   True
 test Hungarian (max total Dice) 0.5165 0.5436     0.0271 [+0.0128, +0.0421]   True
  oof       greedy Dice (thesis) 0.4653 0.4631    -0.0023 [-0.0121, +0.0077]  False
  oof          greedy IoU >= 0.1 0.4642 0.4620    -0.0021 [-0.0119, +0.0079]  False
  oof Hungarian (max total Dice) 0.4655 0.4631    -0.0024 [-0.0122, +0.0076]  False

self-check: greedy-Dice rule on test gives B = 0.5165, N = 0.5436 (thesis: 0.5165 and 0.5436) -> OK


In [11]:
# ============================================================
# CHECK 2) Test-vs-OOF gap: split by lesions per image and by lesion area; re-weight the OOF to the test mix
# ============================================================
def img_info(items):
    n = np.array([len(it["gts"]) for it in items]); area = np.array([sum(g.area for g in it["gts"]) for it in items]); return n, area
n_t, a_t = img_info(TEST_ITEMS); n_o, a_o = img_info(POOL_ITEMS)
cut = np.percentile(np.concatenate([a_o, a_t]), [100 / 3, 200 / 3])          # area groups: thirds of all 1500 images
g_cnt = lambda n: np.minimum(n, 3)                                           # 1, 2, 3+ lesions
g_area = lambda a: np.digitize(a, cut)                                       # 0 small, 1 medium, 2 large
rows = []
for key in ["B", "N"]:
    for metric, name in [("p1", "official-style F1"), ("dice", "Dice/img")]:
        s_t, s_o = OWN[("test", key, "balanced-DEPLOY")][metric], OWN[("oof", key, "balanced-DEPLOY")][metric]
        for gname, gf, ntt, noo, labs in [("lesions per image", g_cnt, n_t, n_o, [1, 2, 3]), ("total lesion area (thirds)", g_area, a_t, a_o, [0, 1, 2])]:
            gt_, go_ = gf(ntt), gf(noo); gap_total = s_t.mean() - s_o.mean(); rew = 0.0
            for l in labs:
                mt, mo = gt_ == l, go_ == l
                rows.append(dict(model=key, metric=name, grouping=gname, group=l, n_oof=int(mo.sum()), n_test=int(mt.sum()),
                                 oof=round(float(s_o[mo].mean()), 4) if mo.any() else None, test=round(float(s_t[mt].mean()), 4) if mt.any() else None))
                if mo.any(): rew += mt.mean() * s_o[mo].mean()
            rows.append(dict(model=key, metric=name, grouping=gname, group="ALL", n_oof=len(s_o), n_test=len(s_t), oof=round(float(s_o.mean()), 4), test=round(float(s_t.mean()), 4),
                             oof_reweighted_to_test_mix=round(float(rew), 4), gap=round(float(gap_total), 4), gap_left_after_reweighting=round(float(s_t.mean() - rew), 4)))
T4g = pd.DataFrame(rows); T4g.to_csv(OUT_DIR / "C2_gap_by_group.csv", index=False)
print(T4g[T4g.group.astype(str) == "ALL"].to_string(index=False)); print("\n(full table with every group in C2_gap_by_group.csv)")

model            metric                   grouping group  n_oof  n_test    oof   test  oof_reweighted_to_test_mix    gap  gap_left_after_reweighting
    B official-style F1          lesions per image   ALL   1200     300 0.4653 0.5165                      0.5125 0.0512                      0.0040
    B official-style F1 total lesion area (thirds)   ALL   1200     300 0.4653 0.5165                      0.4594 0.0512                      0.0571
    B          Dice/img          lesions per image   ALL   1200     300 0.5941 0.6197                      0.5948 0.0257                      0.0249
    B          Dice/img total lesion area (thirds)   ALL   1200     300 0.5941 0.6197                      0.6060 0.0257                      0.0137
    N official-style F1          lesions per image   ALL   1200     300 0.4631 0.5436                      0.5119 0.0805                      0.0317
    N official-style F1 total lesion area (thirds)   ALL   1200     300 0.4631 0.5436                     

In [12]:
# ============================================================
# CHECK 3) One fold model vs the 5-model mean on the test set, at the frozen deploy post-processing
# ============================================================
rows = []
for key, aux, refine in [("B", False, False), ("N", True, True)]:
    op = FROZEN_OPS[key]["balanced-DEPLOY"]
    for f in range(5):
        net = VGNet(aux, refine); net.load_state_dict(torch.load(CK[key][f], map_location="cpu", weights_only=True), strict=True); net.to(DEVICE).eval()
        pf = predict_ms(net, TEST_IMGS); del net
        if USE_CUDA: torch.cuda.empty_cache()
        s = score(pf, GTU_TEST, TEST_ITEMS, op)
        rows.append(dict(model=key, which=f"fold model {f}", dice=round(float(s["dice"].mean()), 4), official_style_F1=round(float(s["p1"].mean()), 4), true_F1=round(float(s["tf1"].mean()), 4), preds_per_image=round(float(s["npred"].mean()), 3)))
    se = OWN[("test", key, "balanced-DEPLOY")]
    rows.append(dict(model=key, which="5-model mean (thesis)", dice=round(float(se["dice"].mean()), 4), official_style_F1=round(float(se["p1"].mean()), 4), true_F1=round(float(se["tf1"].mean()), 4), preds_per_image=round(float(se["npred"].mean()), 3)))
T5 = pd.DataFrame(rows); T5.to_csv(OUT_DIR / "C3_single_fold_vs_ensemble.csv", index=False); print(T5.to_string(index=False))
for key in ["B", "N"]:
    t = T5[(T5.model == key) & T5.which.str.startswith("fold")]
    print(f"{key}: single fold models mean Dice {t.dice.mean():.4f} (min {t.dice.min():.4f}, max {t.dice.max():.4f}) | official-style F1 mean {t.official_style_F1.mean():.4f} (min {t.official_style_F1.min():.4f}, max {t.official_style_F1.max():.4f})")

model                 which   dice  official_style_F1  true_F1  preds_per_image
    B          fold model 0 0.6083             0.5035   0.5455            1.463
    B          fold model 1 0.6043             0.5029   0.5455            1.453
    B          fold model 2 0.5942             0.4873   0.5350            1.530
    B          fold model 3 0.6036             0.4925   0.5395            1.520
    B          fold model 4 0.6028             0.5019   0.5453            1.523
    B 5-model mean (thesis) 0.6197             0.5165   0.5587            1.443
    N          fold model 0 0.6215             0.5231   0.5617            1.443
    N          fold model 1 0.6226             0.5261   0.5648            1.447
    N          fold model 2 0.6126             0.5039   0.5493            1.493
    N          fold model 3 0.6156             0.5029   0.5472            1.513
    N          fold model 4 0.6134             0.4959   0.5441            1.527
    N 5-model mean (thesis) 0.6358      

In [13]:
shutil.make_archive(str(BASE / "inference_checks_3a"), "zip", OUT_DIR)
print("Errors:", ERRORS if ERRORS else "none")
print("Done. Download /kaggle/working/inference_checks_3a.zip (Output tab) and send it to Claude.")

Errors: none
Done. Download /kaggle/working/inference_checks_3a.zip (Output tab) and send it to Claude.
